# Experimento 4B: PubMedBERT + focal loss

**Objetivo:** aislar el efecto de cambiar la funcion de perdida de
`CrossEntropyLoss` a `FocalLoss` (gamma=2.0, la formula estandar del paper
original de Focal Loss, Lin et al. 2017), manteniendo todo lo demas igual
-- mismo neg_ratio=3, mismos datos, mismos hiperparametros. Se construye
**sobre el baseline ya arreglado (3A)** -- `fix_entity_markers()` sigue
aplicado, es la base mas correcta disponible ahora mismo (ver
`HALLAZGOS-BUGS-TOKENIZACION.md`).

**Por que focal loss:** reduce el peso de los ejemplos "faciles" (donde el
modelo ya acierta con confianza) y concentra el gradiente en los dificiles --
pensado para el mismo problema que neg_ratio ataca desde otro angulo (el
75% de las instancias son `no_relation`, la mayoria triviales de clasificar
una vez el modelo aprende lo basico).

**Formula:** `FL(p_t) = -(1-p_t)^gamma * log(p_t)` -- igual que
CrossEntropyLoss pero multiplicada por `(1-p_t)^gamma`, que vale ~0 cuando el
modelo ya acierta con confianza (p_t alto) y ~1 cuando falla o duda.

La comparacion se hace contra **3A** (PubMedBERT + bugs arreglados, sin
typed markers, neg_ratio=3): argmax=0.3338, calibrado=0.4298 @ threshold=0.996.

## 1. Setup

In [1]:
# Ejecucion en servidor local (zape), entorno conda "tfg". Mismo patron que 1G/2A/2B/3A/4A.
import os
HF_CACHE_DIR = os.path.expanduser("~/hf_cache")
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import huggingface_hub.constants as hfc
assert hfc.HF_HUB_CACHE == os.environ["HF_HUB_CACHE"], (
    "Reinicia el kernel y ejecuta esta celda ANTES de cualquier import de HF/opennre.")
print("HF cache:", os.environ["HF_HUB_CACHE"])


HF cache: /home/lucia.esperon/hf_cache/hub


In [2]:
import json, time, logging, gc, random
from collections import Counter
from pathlib import Path

import nltk, pandas as pd, torch, numpy as np
import torch.nn as nn

logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import opennre

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

import sys
sys.path.insert(0, "../baseline")
from patch_opennre import add_macro_f1_metric, fix_entity_markers
add_macro_f1_metric()
fix_entity_markers()   # <-- los dos fixes de HALLAZGOS-BUGS-TOKENIZACION.md, igual que 3A/4A
from score import evaluate

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")


/home/lucia.esperon/miniconda3/envs/tfg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.6.0+cu124 | CUDA: True
GPU: NVIDIA GeForce RTX 2080 Ti (11.5 GB)


## 2. Focal loss

In [3]:
class FocalLoss(nn.Module):
    """FL(p_t) = -(1-p_t)^gamma * log(p_t), gamma=2.0 (Lin et al. 2017)."""
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(weight=weight, reduction="none")

    def forward(self, logits, target):
        ce_loss = self.ce(logits, target)
        pt = torch.exp(-ce_loss)
        focal = ((1 - pt) ** self.gamma) * ce_loss
        return focal.mean()


# verificacion rapida: con gamma=0 debe coincidir exactamente con CrossEntropyLoss
_logits = torch.randn(8, 15)
_target = torch.randint(0, 15, (8,))
_fl_gamma0 = FocalLoss(gamma=0.0)(_logits, _target)
_ce = nn.CrossEntropyLoss()(_logits, _target)
assert torch.allclose(_fl_gamma0, _ce, atol=1e-6), "FocalLoss(gamma=0) deberia ser igual a CrossEntropyLoss"
print(f"OK: FocalLoss(gamma=0) == CrossEntropyLoss ({_fl_gamma0.item():.4f} == {_ce.item():.4f})")

GAMMA = 2.0
print(f"FocalLoss(gamma={GAMMA}) lista para usar.")


OK: FocalLoss(gamma=0) == CrossEntropyLoss (2.9254 == 2.9254)
FocalLoss(gamma=2.0) lista para usar.


## 3. Configuracion

In [4]:
MODEL_NAME      = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
EXPERIMENT_NAME = "pubmedbert_focalloss"
TECHNIQUE       = f"FocalLoss(gamma={GAMMA}) en vez de CrossEntropyLoss -- sobre fix_entity_markers(), unico cambio vs 3A"

MAX_LENGTH     = 256
BATCH_SIZE     = 16
LEARNING_RATE  = 2e-5
EPOCHS         = 15
WARMUP_STEPS   = 300
SEED           = 42
GRAD_CLIP_NORM = 1.0

DATA_DIR    = Path("../data/english")
TRAIN_DATA  = DATA_DIR / "eng_train.txt"   # neg_ratio=3, sin cambios -- solo cambia la loss
DEV_DATA    = DATA_DIR / "eng_dev.txt"
REL2ID_PATH = DATA_DIR / "rel2id.json"
for p in (TRAIN_DATA, DEV_DATA, REL2ID_PATH):
    assert p.exists(), f"FALTA {p}"

with open(REL2ID_PATH) as f:
    rel2id = json.load(f)
id2rel = {v: k for k, v in rel2id.items()}
NO_REL_ID = rel2id["no_relation"]
print(f"Clases: {len(rel2id)} | modelo: {MODEL_NAME}")

OUT_DIR = Path(f"../outputs/4B-pubmedbert-focalloss/seed{SEED}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = OUT_DIR / f"eng_{EXPERIMENT_NAME}.pth.tar"
print("Salida:", OUT_DIR)


Clases: 15 | modelo: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Salida: ../outputs/4B-pubmedbert-focalloss/seed42


## 4. Entrenamiento -- misma funcion que 1G/2A/2B/3A/4A (`train_with_history`)

In [5]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


from opennre.framework.utils import AverageMeter
from tqdm import tqdm

def train_with_history(fw, max_epoch, metric="macro_f1"):
    history, best_metric = [], 0
    for epoch in range(max_epoch):
        fw.train()
        avg_loss, avg_acc = AverageMeter(), AverageMeter()
        t = tqdm(fw.train_loader, desc=f"Epoch {epoch}")
        for data in t:
            if torch.cuda.is_available():
                for i in range(len(data)):
                    try: data[i] = data[i].cuda()
                    except Exception: pass
            label, args = data[0], data[1:]
            logits = fw.parallel_model(*args)
            loss = fw.criterion(logits, label)
            _, pred = logits.max(-1)
            acc = float((pred == label).long().sum()) / label.size(0)
            avg_loss.update(loss.item(), 1); avg_acc.update(acc, 1)
            t.set_postfix(loss=avg_loss.avg, acc=avg_acc.avg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(fw.model.parameters(), GRAD_CLIP_NORM)
            fw.optimizer.step()
            if fw.scheduler is not None: fw.scheduler.step()
            fw.optimizer.zero_grad()
        val = fw.eval_model(fw.val_loader)
        rec = {"epoch": epoch, "train_loss": avg_loss.avg, "train_acc": avg_acc.avg,
               "val_acc": val["acc"], "val_micro_p": val["micro_p"], "val_micro_r": val["micro_r"],
               "val_micro_f1": val["micro_f1"], "val_macro_f1": val["macro_f1"]}
        history.append(rec)
        print(f"Epoch {epoch}: loss={rec['train_loss']:.4f} "
              f"val_micro_f1={rec['val_micro_f1']:.4f} val_macro_f1={rec['val_macro_f1']:.4f}")
        if val[metric] > best_metric:
            print(f"  -> nuevo mejor {metric}={val[metric]:.4f}, guardando checkpoint")
            folder = "/".join(fw.ckpt.split("/")[:-1])
            if folder and not os.path.exists(folder): os.makedirs(folder, exist_ok=True)
            torch.save({"state_dict": fw.model.state_dict()}, fw.ckpt)
            best_metric = val[metric]
    print(f"Mejor {metric} en val: {best_metric:.4f}")
    return history


In [6]:
set_seed(SEED)

encoder = opennre.encoder.BERTEntityEncoder(max_length=MAX_LENGTH, pretrain_path=MODEL_NAME)
model = opennre.model.SoftmaxNN(sentence_encoder=encoder, num_class=len(rel2id), rel2id=rel2id)
framework = opennre.framework.SentenceRE(
    model=model, train_path=str(TRAIN_DATA), val_path=str(DEV_DATA), test_path=str(DEV_DATA),
    ckpt=str(CKPT_PATH), batch_size=BATCH_SIZE, max_epoch=EPOCHS, lr=LEARNING_RATE,
    opt="adamw", warmup_step=WARMUP_STEPS)

framework.criterion = FocalLoss(gamma=GAMMA)   # <-- unico cambio real vs 3A
print(f"criterion reemplazado: {framework.criterion}")

n_params = sum(p.numel() for p in model.parameters())
print(f"Parametros: {n_params:,}")

t0 = time.time()
history = train_with_history(framework, EPOCHS, metric="macro_f1")
train_minutes = (time.time() - t0) / 60
with open(OUT_DIR / f"history_{EXPERIMENT_NAME}.json", "w") as f:
    json.dump(history, f, indent=2)
best = max(history, key=lambda h: h["val_macro_f1"])
macro_f1_curado = best["val_macro_f1"]
print(f"\nEntreno: {train_minutes:.1f} min | mejor epoch={best['epoch']} macro_f1_curado(dev)={macro_f1_curado:.4f}")


2026-07-29 15:00:08,700 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


2026-07-29 15:00:08,727 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/e1354b7a3a09615f6aba48dfad4b7a613eef7062/config.json "HTTP/1.1 200 OK"


2026-07-29 15:00:08,867 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"


2026-07-29 15:00:08,868 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


2026-07-29 15:00:09,007 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"


...

 99%|█████████▉| 185/186 [00:13<00:00, 13.46it/s, acc=0.76]

 99%|█████████▉| 185/186 [00:13<00:00, 13.46it/s, acc=0.76]

100%|██████████| 186/186 [00:13<00:00, 13.47it/s, acc=0.76]


2026-07-29 15:52:05,925 - root - INFO - Evaluation result: {'acc': 0.7603640040444893, 'micro_p': 0.8660268714011516, 'micro_r': 0.7603640040444893, 'micro_f1': 0.8097631012203875}.


Epoch 14: loss=0.0026 val_micro_f1=0.8098 val_macro_f1=0.7454
Mejor macro_f1 en val: 0.7575

Entreno: 51.9 min | mejor epoch=13 macro_f1_curado(dev)=0.7575


## 5. Inferencia en blind + evaluacion oficial (argmax)

In [7]:
BLIND_DEV_PATH = DATA_DIR / "eng_dev_blind.txt"
BLIND_GOLD_TSV = DATA_DIR / "eng-dev-rel.tsv"
for p in (BLIND_DEV_PATH, BLIND_GOLD_TSV):
    assert p.exists(), f"FALTA {p}"

blind_raw = [json.loads(l) for l in open(BLIND_DEV_PATH, encoding="utf-8") if l.strip()]
gold_df_blind = pd.read_csv(BLIND_GOLD_TSV, sep="\t")
print(f"candidatos blind: {len(blind_raw)} | gold real: {len(gold_df_blind)}")

model.load_state_dict(torch.load(str(CKPT_PATH), map_location="cpu")["state_dict"])
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device); model.eval()

t0 = time.time()
all_probs = np.zeros((len(blind_raw), len(rel2id)), dtype=np.float32)
with torch.no_grad():
    for s in range(0, len(blind_raw), 64):
        batch = blind_raw[s:s + 64]
        tok = [encoder.tokenize({"text": i["text"], "h": {"pos": i["h"]["pos"]},
                                 "t": {"pos": i["t"]["pos"]}}) for i in batch]
        fields = [torch.cat([t[k] for t in tok], dim=0).to(device) for k in range(len(tok[0]))]
        logits = model(*fields)
        all_probs[s:s + len(batch)] = torch.softmax(logits, dim=-1).cpu().numpy()
blind_minutes = (time.time() - t0) / 60
np.save(OUT_DIR / f"blind_probs_{EXPERIMENT_NAME}.npy", all_probs)
print(f"Inferencia blind: {blind_minutes:.1f} min")

def rows_from_preds(pred_ids):
    labels = [id2rel[i] for i in pred_ids]
    return [
        {"document_id": inst["doc_id"], "relation": rel,
         "head_text": inst["h"]["name"], "head_span": inst["head_span"], "head_type": inst["head_type"],
         "tail_text": inst["t"]["name"], "tail_span": inst["tail_span"], "tail_type": inst["tail_type"]}
        for inst, rel in zip(blind_raw, labels) if rel != "no_relation"
    ]

argmax_ids = all_probs.argmax(axis=1)
res_argmax = evaluate(pd.DataFrame(rows_from_preds(argmax_ids)), gold_df_blind)
macro_f1_ciego_argmax = res_argmax["macro_f1"]
print(f"Macro F1 ciego (argmax puro): {macro_f1_ciego_argmax:.4f}")


candidatos blind: 282364 | gold real: 2891


Inferencia blind: 22.3 min


Macro F1 ciego (argmax puro): 0.3351


## 6. Calibracion de threshold (grid fino) -- comparacion justa contra 3A

In [8]:
def preds_at_threshold(probs, threshold):
    probs2 = probs.copy()
    probs2[:, NO_REL_ID] = -1
    best_rel_id = probs2.argmax(axis=1)
    best_rel_prob = probs[np.arange(len(probs)), best_rel_id]
    return np.where(best_rel_prob >= threshold, best_rel_id, NO_REL_ID)

def f1_at(probs, threshold):
    pred_ids = preds_at_threshold(probs, threshold)
    return evaluate(pd.DataFrame(rows_from_preds(pred_ids)), gold_df_blind)["macro_f1"]

FINE_GRID = [round(float(x), 4) for x in np.arange(0.85, 0.9901, 0.001)] + \
            [round(float(x), 4) for x in np.arange(0.990, 0.9991, 0.001)] + \
            [0.9995, 0.9999]
FINE_GRID = sorted(set(FINE_GRID))

sweep = [(th, f1_at(all_probs, th)) for th in FINE_GRID]
best_threshold, macro_f1_ciego_calibrado = max(sweep, key=lambda x: x[1])
en_borde = best_threshold == FINE_GRID[-1]
print(f"Mejor threshold (fino): {best_threshold:.4f} -> Macro F1 ciego calibrado = {macro_f1_ciego_calibrado:.4f}"
      f"{'  [BORDE DEL GRID -- revisar]' if en_borde else ''}")

# --- referencia: 3A (PubMedBERT + bugs arreglados, neg_ratio=3, sin typed markers) ---
BASELINE_ARGMAX = 0.333788238025547        # outputs/3A-pubmedbert-fixed/seed42/results_seed_summary.json
BASELINE_CALIBRADO = 0.42979245116674064   # outputs/3A-pubmedbert-fixed/seed42/results_seed_summary.json
BASELINE_TH = 0.996

print(f"\n{'':<28}{'argmax':>10}{'calibrado':>12}{'threshold':>12}")
print(f"{'3A (bugs arreglados, CE loss)':<28}{BASELINE_ARGMAX:>10.4f}{BASELINE_CALIBRADO:>12.4f}{BASELINE_TH:>12.3f}")
print(f"{'+ focal loss (gamma=2)':<28}{macro_f1_ciego_argmax:>10.4f}{macro_f1_ciego_calibrado:>12.4f}{best_threshold:>12.3f}")
print(f"{'delta':<28}{macro_f1_ciego_argmax-BASELINE_ARGMAX:>+10.4f}{macro_f1_ciego_calibrado-BASELINE_CALIBRADO:>+12.4f}")


Mejor threshold (fino): 0.9230 -> Macro F1 ciego calibrado = 0.4371

                                argmax   calibrado   threshold
3A (bugs arreglados, CE loss)    0.3338      0.4298       0.996
+ focal loss (gamma=2)          0.3351      0.4371       0.923
delta                          +0.0013     +0.0073


## 7. Guardar resultados

In [9]:
results = {
    "exp": EXPERIMENT_NAME,
    "model": MODEL_NAME,
    "technique": TECHNIQUE,
    "seed": SEED,
    "hyperparameters": {
        "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS, "warmup_steps": WARMUP_STEPS, "neg_ratio": 3, "seed": SEED, "focal_gamma": GAMMA,
    },
    "macro_f1_curado": macro_f1_curado,
    "macro_f1_ciego_argmax": macro_f1_ciego_argmax,
    "best_threshold_fino": best_threshold,
    "macro_f1_ciego_calibrado": macro_f1_ciego_calibrado,
    "en_borde_del_grid_fino": en_borde,
    "train_minutes": round(train_minutes, 1),
    "blind_minutes": round(blind_minutes, 1),
    "baseline_comparison": {
        "source": "outputs/3A-pubmedbert-fixed/seed42/results_seed_summary.json (bugs arreglados, CrossEntropyLoss)",
        "baseline_macro_f1_ciego_argmax": BASELINE_ARGMAX,
        "baseline_macro_f1_ciego_calibrado": BASELINE_CALIBRADO,
        "delta_argmax": macro_f1_ciego_argmax - BASELINE_ARGMAX,
        "delta_calibrado": macro_f1_ciego_calibrado - BASELINE_CALIBRADO,
    },
}
with open(OUT_DIR / "results_seed_summary.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Guardado:", OUT_DIR / "results_seed_summary.json")
print(json.dumps(results, indent=2, ensure_ascii=False))


Guardado: ../outputs/4B-pubmedbert-focalloss/seed42/results_seed_summary.json
{
  "exp": "pubmedbert_focalloss",
  "model": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
  "technique": "FocalLoss(gamma=2.0) en vez de CrossEntropyLoss -- sobre fix_entity_markers(), unico cambio vs 3A",
  "seed": 42,
  "hyperparameters": {
    "max_length": 256,
    "batch_size": 16,
    "learning_rate": 2e-05,
    "epochs": 15,
    "warmup_steps": 300,
    "neg_ratio": 3,
    "seed": 42,
    "focal_gamma": 2.0
  },
  "macro_f1_curado": 0.7575306164521571,
  "macro_f1_ciego_argmax": 0.33513620451684395,
  "best_threshold_fino": 0.923,
  "macro_f1_ciego_calibrado": 0.43710973609659703,
  "en_borde_del_grid_fino": false,
  "train_minutes": 51.9,
  "blind_minutes": 22.3,
  "baseline_comparison": {
    "source": "outputs/3A-pubmedbert-fixed/seed42/results_seed_summary.json (bugs arreglados, CrossEntropyLoss)",
    "baseline_macro_f1_ciego_argmax": 0.333788238025547,
    "baseline_macro_f1_

## 8. Conclusion

**Resultado: mejora pequena y consistente en direccion (+0.0013 argmax,
+0.0073 calibrado)**, pero el delta calibrado sigue dentro del rango de
ruido de seed a seed medido en el multiseed (std~0.006-0.012) -- no es tan
grande como el de neg_ratio 1:1 (concluyente) ni tan ambiguo como el de
typed markers (signo mixto entre metricas). Con una sola seed, "probablemente
ayuda un poco, pero no se puede afirmar con seguridad" es la lectura honesta.

**Dato interesante:** el mejor threshold baja de 0.996 a 0.923 -- coherente
con lo que se espera de focal loss: al penalizar menos los ejemplos faciles,
el modelo deja de estar tan sobreconfiado, y las probabilidades ya no se
amontonan tan cerca de 1.0, así que el umbral optimo se relaja.